# Notebook 3: Finding the Refusal Direction

**Goal:** Compute the "refusal direction" — the specific vector in the model's activation space that encodes refusal behavior.

**The idea:** If we average the activations for harmful prompts and subtract the average for harmless prompts, what's left should be the component unique to refusal.

## Quick Concept: Projection & Orthogonalization

**Projection** = how much of vector A points in the direction of vector B.

Imagine you're walking northeast. Your "northward component" is the projection of your movement onto the north direction. Your "eastward component" is the projection onto east.

**Orthogonalization** = removing a specific direction from a vector.

If you remove the "north" component from your northeast walk, you're left with only the "east" part.

That's exactly what we'll do: find the "refusal direction", then remove it from the model's weights.

In [ ]:
# Visual demo: projection and orthogonalization in 2D
import numpy as np
import matplotlib.pyplot as plt

# A vector and a direction we want to remove
vector = np.array([3.0, 4.0])
direction = np.array([1.0, 0.0])  # "east" direction

# Projection: how much of 'vector' is in the 'direction'
projection_scalar = np.dot(vector, direction) / np.dot(direction, direction)
projection_vector = projection_scalar * direction

# Orthogonalized: remove the projection
orthogonalized = vector - projection_vector

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.arrow(0, 0, vector[0], vector[1], head_width=0.15, head_length=0.15, fc='blue', ec='blue', linewidth=2)
ax.arrow(0, 0, projection_vector[0], projection_vector[1], head_width=0.15, head_length=0.15, fc='red', ec='red', linewidth=2)
ax.arrow(0, 0, orthogonalized[0], orthogonalized[1], head_width=0.15, head_length=0.15, fc='green', ec='green', linewidth=2)

# Draw dashed line showing the removal
ax.plot([vector[0], orthogonalized[0]], [vector[1], orthogonalized[1]], 'k--', alpha=0.3)

ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(['Original [3,4]', 'Projection onto east [3,0]', 'After removing east [0,4]'], fontsize=10)
ax.set_title("Orthogonalization: removing a direction from a vector")
plt.tight_layout()
plt.show()

print(f"Original vector:      {vector}")
print(f"Direction to remove:  {direction} (east)")
print(f"Projection:           {projection_vector} (the 'east' component)")
print(f"After removal:        {orthogonalized} (only 'north' remains!)")

## Step 1: Load Activations from Notebook 2

In [ ]:
import torch
import json
import numpy as np
import matplotlib.pyplot as plt

torch.set_grad_enabled(False)

# Load activations saved from notebook 2
harmless_activations = torch.load("data/harmless_activations.pt", weights_only=True)
harmful_activations = torch.load("data/harmful_activations.pt", weights_only=True)

with open("data/model_info.json", "r") as f:
    model_info = json.load(f)

n_layers = model_info["n_layers"]
d_model = model_info["d_model"]

print(f"Model: {model_info['model_name']}")
print(f"Layers: {n_layers}, Hidden dim: {d_model}")
print(f"Activations loaded: {len(harmless_activations)} hooks")

## Step 2: Compute Refusal Direction Candidates

For each layer, we:
1. Compute the **mean activation** for harmful prompts
2. Compute the **mean activation** for harmless prompts
3. Take the **difference** → this is our candidate refusal direction for that layer
4. **Normalize** it (make it unit length) so we can compare across layers

In [ ]:
# Compute refusal direction for each layer
refusal_directions = {}

for layer in range(n_layers):
    hook_name = f"blocks.{layer}.hook_resid_pre"
    
    # Mean activation across all samples: (d_model,)
    mean_harmful = harmful_activations[hook_name].float().mean(dim=0)
    mean_harmless = harmless_activations[hook_name].float().mean(dim=0)
    
    # Difference = candidate refusal direction
    diff = mean_harmful - mean_harmless
    
    # Normalize to unit length
    refusal_dir = diff / diff.norm()
    
    refusal_directions[layer] = refusal_dir

print(f"Computed {len(refusal_directions)} refusal direction candidates (one per layer)")
print(f"Each direction is a {d_model}-dimensional unit vector")

## Step 3: Score the Candidates — Which Layer Has the Strongest Refusal Signal?

Not all layers encode refusal equally. We score each direction by how well it separates harmful from harmless activations — specifically, the **mean absolute projection** of each sample onto the candidate direction.

In [ ]:
# Score each layer's refusal direction
# A good refusal direction should:
#   - have HIGH projection for harmful prompts (model "activates" refusal)
#   - have LOW projection for harmless prompts (no refusal needed)
# So we look at the DIFFERENCE in mean projections

scores = {}

for layer in range(n_layers):
    hook_name = f"blocks.{layer}.hook_resid_pre"
    direction = refusal_directions[layer]
    
    # Project each sample onto the refusal direction (dot product)
    # Higher value = more aligned with refusal
    harmful_projections = (harmful_activations[hook_name].float() @ direction).mean().item()
    harmless_projections = (harmless_activations[hook_name].float() @ direction).mean().item()
    
    # Score = how much more the harmful prompts align with this direction
    scores[layer] = harmful_projections - harmless_projections

# Plot scores
plt.figure(figsize=(12, 5))
plt.bar(range(n_layers), [scores[i] for i in range(n_layers)], color='steelblue')
plt.xlabel('Layer')
plt.ylabel('Separation Score (harmful - harmless projection)')
plt.title('Refusal Direction Strength per Layer')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Find the best layer
best_layer = max(scores, key=scores.get)
print(f"\nBest layer: {best_layer} (score: {scores[best_layer]:.4f})")
print(f"This layer's refusal direction best separates harmful from harmless prompts.")

## Step 4: Visualize the Separation

Let's see how well the best refusal direction separates harmful from harmless activations. We'll plot the projection values for each sample — if there's a clear gap, we've found a good refusal direction.

In [ ]:
# Plot histogram of projections for the best layer
hook_name = f"blocks.{best_layer}.hook_resid_pre"
direction = refusal_directions[best_layer]

harmful_proj = (harmful_activations[hook_name].float() @ direction).numpy()
harmless_proj = (harmless_activations[hook_name].float() @ direction).numpy()

plt.figure(figsize=(10, 5))
plt.hist(harmless_proj, bins=40, alpha=0.6, label='Harmless', color='blue')
plt.hist(harmful_proj, bins=40, alpha=0.6, label='Harmful', color='red')
plt.xlabel('Projection onto refusal direction')
plt.ylabel('Count')
plt.title(f'Layer {best_layer}: How much each prompt aligns with the refusal direction')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("If the red (harmful) and blue (harmless) distributions are separated,")
print("we've found a direction that the model uses to distinguish 'should I refuse?'")

## Step 5: Save the Refusal Direction

In [ ]:
# Save the best refusal direction and all candidates
save_data = {
    "best_layer": best_layer,
    "scores": scores,
    "refusal_directions": {layer: direction for layer, direction in refusal_directions.items()},
}
torch.save(save_data, "data/refusal_directions.pt")

print(f"Saved refusal directions to data/refusal_directions.pt")
print(f"Best layer: {best_layer}")
print(f"\nTop 5 layers by score:")
sorted_layers = sorted(scores.items(), key=lambda x: x[1], reverse=True)
for layer, score in sorted_layers[:5]:
    print(f"  Layer {layer}: {score:.4f}")

## Summary

In this notebook we:
1. Learned **projection** (how much a vector aligns with a direction) and **orthogonalization** (removing a direction)
2. Computed refusal direction candidates for every layer: `mean(harmful) - mean(harmless)`
3. Scored each layer by how well its direction separates harmful from harmless prompts
4. Found the **best layer** and visualized the separation

**Next → Notebook 04:** We'll test if removing this direction actually stops the model from refusing — first temporarily (via hooks), before making it permanent.

---

**Try it yourself:** Look at the bar chart of scores. Do the early layers or later layers have stronger refusal signals? Why might that be? (Hint: early layers process low-level features, later layers process high-level concepts like "this is harmful")